# Extract Rust Starter Code from Local Files

This notebook reads Rust `.rs` files from the local `rust_solutions/` directory,
extracts starter code stubs (comments + `impl Solution` with function bodies replaced by placeholder), and saves to `starter_codes.jsonl`.

Expected file format: `0001.Two Sum.rs` where the leading digits are the problem_id.

Output format:
```rust
// Definition for singly-linked list.
// #[derive(PartialEq, Eq, Clone, Debug)]
// pub struct ListNode { ... }

impl Solution {
    pub fn add_two_numbers(
        mut l1: Option<Box<ListNode>>,
        mut l2: Option<Box<ListNode>>,
    ) -> Option<Box<ListNode>> {
        // OUR CODE GOES HERE
    }
}
```

In [ ]:
import json
import re
from pathlib import Path

RUST_SOLUTIONS_DIR = Path("/media/tm23/NewVolume/rust_dataset/complexity_reasoning_data/rust_solutions/")
OUTPUT_FILE = Path("/media/tm23/NewVolume/rust_dataset/complexity_reasoning_data/starter_codes.jsonl")

print(f"Reading from: {RUST_SOLUTIONS_DIR}")
print(f"Output to: {OUTPUT_FILE}")

In [ ]:
def find_matching_brace(text: str, open_pos: int) -> int:
    """Find the position of the closing brace matching the opening brace at open_pos."""
    depth = 0
    i = open_pos
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return i
        i += 1
    return -1


def extract_problem_id(filename: str) -> int | None:
    """Extract problem_id from filename like '0001.Two Sum.rs' -> 1"""
    match = re.match(r"^(\d+)", filename)
    return int(match.group(1)) if match else None


def extract_starter_code(content: str) -> tuple[str, str] | None:
    """
    Extract Rust starter code stub from solution.
    Preserves initial comments/struct definitions and impl Solution block
    with function bodies replaced by placeholder comment.
    
    Returns:
        tuple of (function_name, starter_code) or None if no impl Solution found
    """
    if "impl Solution" not in content:
        return None

    impl_match = re.search(r"impl\s+Solution\s*\{", content)
    if not impl_match:
        return None

    impl_start = impl_match.start()
    brace_pos = impl_match.end() - 1

    impl_end = find_matching_brace(content, brace_pos)
    if impl_end == -1:
        return None

    impl_block_content = content[brace_pos + 1:impl_end]

    fn_pattern = re.compile(r"(?:pub\s+)?fn\s+(\w+)\s*\([^)]*\)(?:\s*->\s*[^{]+)?\s*\{")
    fn_matches = list(fn_pattern.finditer(impl_block_content))

    if not fn_matches:
        return None

    pub_fn_match = re.search(r"pub\s+fn\s+(\w+)", impl_block_content)
    first_fn_name = pub_fn_match.group(1) if pub_fn_match else fn_matches[0].group(1)

    starter_impl_content = impl_block_content
    for fn_match in reversed(fn_matches):
        fn_body_start = fn_match.end()
        fn_body_end = find_matching_brace(starter_impl_content, fn_body_start - 1)
        if fn_body_end == -1:
            continue
        replacement = "\n        // OUR CODE GOES HERE\n    }"
        starter_impl_content = starter_impl_content[:fn_body_start] + replacement + starter_impl_content[fn_body_end + 1:]

    pre_impl = content[:impl_start].strip()

    if pre_impl:
        starter_code = pre_impl + "\n\nimpl Solution {" + starter_impl_content + "\n}"
    else:
        starter_code = "impl Solution {" + starter_impl_content + "\n}"

    return first_fn_name, starter_code


def extract_all_starter_codes():
    """Extract starter codes from all .rs files in the directory."""
    results = []
    
    if not RUST_SOLUTIONS_DIR.exists():
        print(f"\u274c Directory does not exist: {RUST_SOLUTIONS_DIR}")
        print("Please place .rs files in this directory with format: 0001.Two Sum.rs")
        return results
    
    rs_files = list(RUST_SOLUTIONS_DIR.glob("*.rs"))
    print(f"Found {len(rs_files)} .rs files")
    
    for rs_file in sorted(rs_files):
        problem_id = extract_problem_id(rs_file.name)
        if problem_id is None:
            print(f"  Skipping (no problem_id): {rs_file.name}")
            continue
        
        content = rs_file.read_text()
        result = extract_starter_code(content)
        
        if result is None:
            print(f"  Skipping (no impl Solution): {rs_file.name}")
            continue
        
        fn_name, starter_code = result
        results.append({
            "problem_id": problem_id,
            "function_name": fn_name,
            "starter_code": starter_code
        })
        print(f"  Extracted: {problem_id} -> {fn_name}()")
    
    return results


results = extract_all_starter_codes()
print(f"\n\u2705 Extracted {len(results)} starter codes")

In [ ]:
with open(OUTPUT_FILE, "w") as f:
    for item in results:
        f.write(json.dumps(item) + "\n")

print(f"\u2705 Saved to {OUTPUT_FILE}")

print("\n" + "="*50)
print("First 3 entries:")
for item in results[:3]:
    print(f"\n--- Problem {item['problem_id']}: {item['function_name']} ---")
    print(item['starter_code'][:600])
    print("...")

In [2]:
import json
def load_ids(path, key="problem_id"):
    ids = set()
    with open(path) as f:
        for line in f:
            data = json.loads(line)
            ids.add(data[key])
    return ids
dataset_ids   = load_ids("/teamspace/studios/this_studio/dataset/complexity_reasoning_data/dataset.jsonl")
starter_ids   = load_ids("/teamspace/studios/this_studio/dataset/complexity_reasoning_data/starter_codes.jsonl")
tests_ids     = load_ids("/teamspace/studios/this_studio/dataset/complexity_reasoning_data/python_tests.jsonl")
missing_starter = dataset_ids - starter_ids
missing_tests   = dataset_ids - tests_ids
will_convert    = dataset_ids & starter_ids & tests_ids
print(f"Dataset IDs:     {len(dataset_ids)}")
print(f"Starter code IDs: {len(starter_ids)}")
print(f"Python test IDs:  {len(tests_ids)}")
print(f"\nMissing starter code ({len(missing_starter)}): {sorted(missing_starter)}")
print(f"\nMissing python tests ({len(missing_tests)}): {sorted(missing_tests)}")
print(f"\nWill be converted ({len(will_convert)}): {sorted(will_convert)}")

Dataset IDs:     2044
Starter code IDs: 1104
Python test IDs:  2641

Missing starter code (1166): [4, 8, 16, 18, 29, 30, 31, 37, 44, 51, 52, 68, 72, 87, 89, 91, 93, 99, 127, 131, 132, 141, 160, 161, 162, 163, 164, 171, 172, 174, 186, 188, 219, 220, 221, 223, 224, 227, 233, 234, 239, 245, 246, 247, 248, 249, 259, 261, 266, 270, 276, 293, 299, 305, 309, 310, 311, 313, 314, 316, 318, 319, 320, 323, 328, 329, 330, 331, 332, 340, 349, 357, 358, 360, 363, 365, 369, 370, 375, 376, 377, 383, 387, 393, 400, 407, 408, 410, 412, 413, 414, 419, 422, 424, 434, 435, 436, 444, 447, 453, 455, 464, 466, 476, 480, 482, 487, 503, 506, 507, 514, 516, 517, 518, 520, 522, 523, 525, 527, 531, 533, 537, 541, 544, 545, 554, 555, 562, 563, 575, 576, 600, 609, 628, 629, 630, 634, 636, 638, 640, 644, 647, 657, 664, 678, 680, 683, 684, 685, 689, 692, 698, 699, 700, 701, 714, 721, 723, 727, 741, 742, 743, 747, 750, 753, 754, 755, 757, 760, 761, 765, 779, 781, 782, 788, 801, 805, 807, 809, 810, 813, 815, 825, 826, 8

In [6]:
import re
from pathlib import Path

# Define the path to your folder
solutions_dir = Path("/teamspace/studios/this_studio/dataset/complexity_reasoning_data/rust_solutions")

# Get all .rs files and sort them
rs_files = sorted(solutions_dir.glob("*.rs"))

# --- Initialize Counters ---
total_files = len(rs_files)
skipped_no_impl = 0
skipped_pattern_mismatch = 0
valid_count = 0

print(f"Checking {total_files} files in {solutions_dir.name}...\n")

for rs_file in rs_files:
    content = rs_file.read_text()
    
    if "impl Solution" not in content:
        impl_matches = re.findall(r"impl\s+(\w+)", content)
        print(f"⚠️  Skipped (no impl Solution): {rs_file.name} — found: {impl_matches}")
        skipped_no_impl += 1
        
    elif not re.search(r"impl\s+Solution\s*\{", content):
        print(f"❓  Skipped (pattern mismatch): {rs_file.name}")
        skipped_pattern_mismatch += 1
        
    else:
        valid_count += 1

# --- Final Summary ---
print("\n" + "="*30)
print("       FINAL SUMMARY")
print("="*30)
print(f"Total Files Scanned:  {total_files}")
print(f"✅ Valid Solutions:    {valid_count}")
print(f"❌ No 'impl Solution': {skipped_no_impl}")
print(f"⚠️  Pattern Mismatch:  {skipped_pattern_mismatch}")
print(f"Total Skipped:        {skipped_no_impl + skipped_pattern_mismatch}")
print("="*30)

Checking 1141 files in rust_solutions...

⚠️  Skipped (no impl Solution): 0146.LRU Cache.rs — found: ['Node', 'LRUCache']
⚠️  Skipped (no impl Solution): 0155.Min Stack.rs — found: ['MinStack']
⚠️  Skipped (no impl Solution): 0173.Binary Search Tree Iterator.rs — found: ['TreeNode', 'BSTIterator']
⚠️  Skipped (no impl Solution): 0208.Implement Trie (Prefix Tree).rs — found: ['TrieNode', 'Trie']
⚠️  Skipped (no impl Solution): 0225.Implement Stack using Queues.rs — found: ['MyStack']
⚠️  Skipped (no impl Solution): 0232.Implement Queue using Stacks.rs — found: ['MyQueue']
⚠️  Skipped (no impl Solution): 0281.Zigzag Iterator.rs — found: ['ZigzagIterator']
⚠️  Skipped (no impl Solution): 0295.Find Median from Data Stream.rs — found: ['MedianFinder']
⚠️  Skipped (no impl Solution): 0303.Range Sum Query - Immutable.rs — found: ['NumArray']
⚠️  Skipped (no impl Solution): 0304.Range Sum Query 2D - Immutable.rs — found: ['NumMatrix']
⚠️  Skipped (no impl Solution): 0341.Flatten Nested List It

In [7]:
import json
import re
from pathlib import Path
RUST_SOLUTIONS_DIR = Path("/media/tm23/NewVolume/rust_dataset/complexity_reasoning_data/rust_solutions/")
files_with_trailing_comments = []
files_with_multiple_impl = []
files_with_multiple_funcs = []
for rs_file in sorted(RUST_SOLUTIONS_DIR.glob("*.rs")):
    content = rs_file.read_text()
    if "impl Solution" not in content:
        continue
    # Check for multiple impl blocks
    impl_blocks = list(re.finditer(r"impl\s+\w+\s*\{", content))
    if len(impl_blocks) > 1:
        impl_names = [re.search(r"impl\s+(\w+)", m.group()).group(1) for m in impl_blocks]
        files_with_multiple_impl.append({
            "file": rs_file.name,
            "impls": impl_names,
        })
    # Check for trailing content after impl Solution block
    impl_match = re.search(r"impl\s+Solution\s*\{", content)
    if impl_match:
        # Find matching closing brace
        depth = 0
        i = impl_match.end() - 1
        while i < len(content):
            if content[i] == '{':
                depth += 1
            elif content[i] == '}':
                depth -= 1
                if depth == 0:
                    impl_end = i
                    break
            i += 1
        else:
            impl_end = len(content)
        trailing = content[impl_end + 1:].strip()
        if trailing:
            # Check if it has actual content (not just whitespace)
            lines = [l for l in trailing.splitlines() if l.strip() and not l.strip().startswith("//")]
            comment_lines = [l for l in trailing.splitlines() if l.strip().startswith("//")]
            if comment_lines or lines:
                files_with_trailing_comments.append({
                    "file": rs_file.name,
                    "trailing": trailing[:300],
                    "has_code": bool(lines),
                    "comment_count": len(comment_lines),
                })
    # Check for multiple functions in impl Solution
    impl_start = impl_match.start() if impl_match else 0
    impl_block = content[impl_start:impl_end + 1] if impl_match else ""
    fn_matches = re.findall(r"(?:pub\s+)?fn\s+(\w+)", impl_block)
    if len(fn_matches) > 1:
        files_with_multiple_funcs.append({
            "file": rs_file.name,
            "functions": fn_matches,
        })
print(f"Files with trailing comments after impl Solution: {len(files_with_trailing_comments)}")
for item in files_with_trailing_comments[:10]:
    print(f"  {item['file']} — {item['comment_count']} comment lines, has_code={item['has_code']}")
    print(f"    Trailing: {item['trailing'][:150]}")
if len(files_with_trailing_comments) > 10:
    print(f"  ... and {len(files_with_trailing_comments) - 10} more")
print(f"\nFiles with multiple impl blocks: {len(files_with_multiple_impl)}")
for item in files_with_multiple_impl[:10]:
    print(f"  {item['file']} — impls: {item['impls']}")
if len(files_with_multiple_impl) > 10:
    print(f"  ... and {len(files_with_multiple_impl) - 10} more")
print(f"\nFiles with multiple functions in impl Solution: {len(files_with_multiple_funcs)}")
for item in files_with_multiple_funcs[:10]:
    print(f"  {item['file']} — functions: {item['functions']}")
if len(files_with_multiple_funcs) > 10:
    print(f"  ... and {len(files_with_multiple_funcs) - 10} more")

Files with trailing comments after impl Solution: 0

Files with multiple impl blocks: 0

Files with multiple functions in impl Solution: 0
